<a href="https://colab.research.google.com/github/hibahrehman25-lang/ML_inter_Task1/blob/main/Copy_of_w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hibahrehman25-lang/ML_inter_Task1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## ML-10 — Content Action Playbook

This notebook turns the validated Week-6 opportunity analysis into a practical, human-reviewed content action playbook.

The queue is intended for prioritization and decision-support. It does not claim that recommended actions will cause additional clicks or traffic.

In [ ]:
# ============================================================
# STEP 0 — Imports and environment setup
# ============================================================

import duckdb
import pandas as pd
import numpy as np
import json
from pathlib import Path

from google.colab import userdata

print("Imports successful")

# Load Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token loaded:", HF_TOKEN is not None)

if HF_TOKEN is None:
    raise ValueError(
        "HF_TOKEN was not found in Colab Secrets. "
        "Add your Hugging Face token as HF_TOKEN and run again."
    )

# Connect to DuckDB
con = duckdb.connect()

# Configure Hugging Face authentication
con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("DuckDB connected and Hugging Face authentication configured.")

Imports successful
HF token loaded: True
DuckDB connected and Hugging Face authentication configured.


## Recreate the validated opportunity dataset

The Week-7 playbook uses the same March 2026 content-level modeling window used in the Week-5/Week-6 analysis.

The dataset is aggregated to one row per client-content pair. The opportunity definition is recreated from the same observed signals so that the Week-7 queue remains consistent with the previous analysis.

In [ ]:
# ============================================================
# STEP 1 — Recreate the Week-5/Week-6 modeling dataset
# ============================================================

model_df = con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS total_impressions,
    SUM(gsc_clicks) AS total_clicks,
    AVG(gsc_avg_position) AS avg_position,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN (SUM(gsc_clicks) * 100.0) / SUM(gsc_impressions)
        ELSE NULL
    END AS ctr

FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'

WHERE
    gsc_data_available = TRUE
    AND gsc_impressions > 0
    AND gsc_avg_position > 0

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

print("Model rows:", len(model_df))
print("Unique clients:", model_df["client_hash_id"].nunique())

print("\nColumns:")
print(model_df.columns.tolist())

print("\nPublic-safe preview:")
display(
    model_df[
        [
            "total_impressions",
            "total_clicks",
            "avg_position",
            "ctr"
        ]
    ].head()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Model rows: 175304
Unique clients: 47

Columns:
['client_hash_id', 'content_hash_id', 'total_impressions', 'total_clicks', 'avg_position', 'ctr']

Public-safe preview:


,total_impressions,total_clicks,avg_position,ctr
0,70.0,0.0,4.888929,0.000000
1,10849.0,22.0,8.240351,0.202784
2,56.0,0.0,7.061594,0.000000
3,47.0,0.0,14.343567,0.000000
4,2099.0,1.0,3.066796,0.047642


In [ ]:
# ============================================================
# STEP 2 — Recreate the Week-5/Week-6 opportunity label
# ============================================================

impression_threshold = model_df["total_impressions"].quantile(0.75)

model_df["opportunity_label"] = (
    (model_df["total_impressions"] >= impression_threshold)
    & (model_df["total_clicks"] == 0)
).astype(int)

print("75th percentile impression threshold:", impression_threshold)

print("\nOpportunity label counts:")
print(model_df["opportunity_label"].value_counts())

print("\nOpportunity label proportions:")
print(
    model_df["opportunity_label"]
    .value_counts(normalize=True)
)

75th percentile impression threshold: 1052.0

Opportunity label counts:
opportunity_label
0    171247
1      4057
Name: count, dtype: int64

Opportunity label proportions:
opportunity_label
0    0.976857
1    0.023143
Name: proportion, dtype: float64


In [ ]:
# ============================================================
# STEP 3 — Basic sanity checks
# ============================================================

assert len(model_df) == 175304
assert model_df["client_hash_id"].nunique() == 47
assert model_df["opportunity_label"].sum() == 4057
assert model_df["opportunity_label"].isna().sum() == 0

print("Sanity checks passed.")
print("Rows:", len(model_df))
print("Clients:", model_df["client_hash_id"].nunique())
print("Opportunities:", model_df["opportunity_label"].sum())
print(
    "Opportunity rate:",
    round(model_df["opportunity_label"].mean(), 6)
)

Sanity checks passed.
Rows: 175304
Clients: 47
Opportunities: 4057
Opportunity rate: 0.023143


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions and reason codes

The action queue is designed as a decision-support tool for human content review.

Pages are prioritized using transparent, observed signals from the March 2026 modeling dataset. The strongest opportunity pattern identified in the previous analysis was high search visibility combined with zero observed clicks.

The queue therefore prioritizes pages that match this pattern and uses reason codes to make the recommendation understandable to a reviewer.

The ranking does not claim that changing a page will increase clicks or traffic. It identifies pages that appear worth reviewing first based on the measured opportunity pattern.

In [ ]:
# ============================================================
# SECTION 1 — Ranked actions + reason codes
# ============================================================

# Create a working copy of the validated modeling dataset.
queue = model_df.copy()

# ------------------------------------------------------------
# 1. Define transparent reason codes
# ------------------------------------------------------------

queue["reason_code"] = np.select(
    [
        # Strongest opportunity pattern from Week 5/6
        (queue["total_impressions"] >= impression_threshold)
        & (queue["total_clicks"] == 0),

        # High visibility with measurable but relatively low CTR
        (queue["total_impressions"] >= impression_threshold)
        & (queue["ctr"] > 0)
        & (queue["ctr"] < queue["ctr"].median()),

        # Other pages with high observed visibility
        (queue["total_impressions"] >= impression_threshold)
    ],
    [
        "HIGH_VISIBILITY_ZERO_CLICKS",
        "HIGH_VISIBILITY_LOW_CTR",
        "HIGH_VISIBILITY_REVIEW"
    ],
    default="MONITOR"
)

# ------------------------------------------------------------
# 2. Assign transparent priority levels
# ------------------------------------------------------------

priority_map = {
    "HIGH_VISIBILITY_ZERO_CLICKS": 1,
    "HIGH_VISIBILITY_LOW_CTR": 2,
    "HIGH_VISIBILITY_REVIEW": 3,
    "MONITOR": 4
}

queue["priority_level"] = queue["reason_code"].map(priority_map)

# ------------------------------------------------------------
# 3. Add human-readable suggested actions
# ------------------------------------------------------------

action_map = {
    "HIGH_VISIBILITY_ZERO_CLICKS":
        "Review search intent, title/snippet, and SERP alignment",

    "HIGH_VISIBILITY_LOW_CTR":
        "Review title/snippet and search-intent alignment",

    "HIGH_VISIBILITY_REVIEW":
        "Review page visibility and query/page alignment",

    "MONITOR":
        "Monitor before making a content change"
}

queue["suggested_action"] = queue["reason_code"].map(action_map)

# ------------------------------------------------------------
# 4. Rank the queue
# ------------------------------------------------------------

# Higher-priority reason codes come first.
# Within each priority, higher impressions come first.
queue = queue.sort_values(
    ["priority_level", "total_impressions"],
    ascending=[True, False]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

# ------------------------------------------------------------
# 5. Sanity checks
# ------------------------------------------------------------

assert len(queue) == len(model_df)
assert queue["reason_code"].isna().sum() == 0
assert queue["suggested_action"].isna().sum() == 0
assert queue["rank"].is_unique

# ------------------------------------------------------------
# 6. Display a public-safe preview
# ------------------------------------------------------------

display(
    queue[
        [
            "rank",
            "priority_level",
            "reason_code",
            "suggested_action",
            "total_impressions",
            "total_clicks",
            "avg_position",
            "ctr"
        ]
    ].head(20)
)

print("\nReason-code counts:")
print(queue["reason_code"].value_counts())

print("\nQueue rows:", len(queue))

,rank,priority_level,reason_code,suggested_action,total_impressions,total_clicks,avg_position,ctr
0,1,1,HIGH_VISIBILITY_ZERO_CLICKS,"Review search intent, title/snippet, and SERP ...",44707.0,0.0,7.906249,0.0
1,2,1,HIGH_VISIBILITY_ZERO_CLICKS,"Review search intent, title/snippet, and SERP ...",38865.0,0.0,5.694764,0.0
2,3,1,HIGH_VISIBILITY_ZERO_CLICKS,"Review search intent, title/snippet, and SERP ...",33867.0,0.0,39.663601,0.0
3,4,1,HIGH_VISIBILITY_ZERO_CLICKS,"Review search intent, title/snippet, and SERP ...",30834.0,0.0,40.214085,0.0
4,5,1,HIGH_VISIBILITY_ZERO_CLICKS,"Review search intent, title/snippet, and SERP ...",28950.0,0.0,9.419929,0.0
5,6,1,HIGH_VISIBILITY_ZERO_CLICKS,"Review search intent, title/snippet, and SERP ...",28934.0,0.0,47.084829,0.0
6,7,1,HIGH_VISIBILITY_ZERO_CLICKS,"Review search intent, title/snippet, and SERP ...",24908.0,0.0,4.009764,0.0
7,8,1,HIGH_VISIBILITY_ZERO_CLICKS,"Review search intent, title/snippet, and SERP ...",23997.0,0.0,20.845697,0.0
8,9,1,HIGH_VISIBILITY_ZERO_CLICKS,"Review search intent, title/snippet, and SERP ...",21939.0,0.0,51.573254,0.0
9,10,1,HIGH_VISIBILITY_ZERO_CLICKS,"Review search intent, title/snippet, and SERP ...",21519.0,0.0,9.092933,0.0



Reason-code counts:
reason_code
MONITOR                        131465
HIGH_VISIBILITY_REVIEW          39782
HIGH_VISIBILITY_ZERO_CLICKS      4057
Name: count, dtype: int64

Queue rows: 175304


In [ ]:
# ============================================================
# SECTION 1B — Inspect available content metadata
# ============================================================

content_columns = con.sql("""
DESCRIBE
SELECT *
FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
""").df()

display(content_columns)

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [ ]:
# ============================================================
# SECTION 1B — Inspect real content archetypes and intents
# ============================================================

content_types = con.sql("""
SELECT
    content_type,
    COUNT(*) AS n
FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
GROUP BY content_type
ORDER BY n DESC
""").df()

print("Content type distribution:")
display(content_types)

print("\nMain intent distribution:")
main_intents = con.sql("""
SELECT
    main_intent,
    COUNT(*) AS n
FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
GROUP BY main_intent
ORDER BY n DESC
""").df()

display(main_intents)

Content type distribution:


,content_type,n
0,keyword article,459174
1,feedly article,57024
2,comparison article,3408



Main intent distribution:


,main_intent,n
0,informational,260233
1,None,148398
2,transactional,55956
3,commercial,52762
4,navigational,2257


### Archetype-to-action mapping

The warehouse provides three observed content types: keyword article, feedly article, and comparison article. These are used as content archetypes in the playbook.

The archetype is used to guide the type of human review rather than to prescribe an automatic change. Main search intent is included as additional context where it is available.

The recommended action remains decision-support: a reviewer should confirm the page's intent, content quality, and business context before making any change.

In [ ]:
# ============================================================
# SECTION 1C — Add real content archetype and search intent
# ============================================================

# Build one metadata row per client-content pair.
content_metadata = con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    ANY_VALUE(content_type) AS content_type,
    ANY_VALUE(main_intent) AS main_intent,
    ANY_VALUE(content_updated_date) AS content_updated_date,
    ANY_VALUE(last_optimized_date) AS last_optimized_date
FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

print("Content metadata rows:", len(content_metadata))

# Join metadata to the ranked queue.
queue = queue.merge(
    content_metadata,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

# ------------------------------------------------------------
# Archetype → human-review action
# ------------------------------------------------------------

archetype_action_map = {
    "keyword article":
        "Review search intent, title/snippet, and informational alignment",

    "feedly article":
        "Review freshness, relevance, title/snippet, and search intent",

    "comparison article":
        "Review comparison intent, completeness, and title/snippet alignment"
}

queue["archetype_action"] = (
    queue["content_type"]
    .map(archetype_action_map)
    .fillna("Review content type and search intent manually")
)

# ------------------------------------------------------------
# Add intent context
# ------------------------------------------------------------

queue["intent_context"] = (
    queue["main_intent"]
    .fillna("unknown")
    .str.lower()
)

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert len(queue) == 175304
assert queue["archetype_action"].isna().sum() == 0

print("\nContent types represented in the queue:")
display(
    queue["content_type"]
    .value_counts(dropna=False)
    .rename_axis("content_type")
    .reset_index(name="n")
)

print("\nArchetype-action preview:")
display(
    queue[
        [
            "rank",
            "reason_code",
            "content_type",
            "main_intent",
            "archetype_action"
        ]
    ].head(20)
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Content metadata rows: 519606

Content types represented in the queue:


,content_type,n
0,keyword article,158954
1,feedly article,12997
2,comparison article,3353



Archetype-action preview:


,rank,reason_code,content_type,main_intent,archetype_action
0,1,HIGH_VISIBILITY_ZERO_CLICKS,keyword article,commercial,"Review search intent, title/snippet, and infor..."
1,2,HIGH_VISIBILITY_ZERO_CLICKS,keyword article,transactional,"Review search intent, title/snippet, and infor..."
2,3,HIGH_VISIBILITY_ZERO_CLICKS,keyword article,informational,"Review search intent, title/snippet, and infor..."
3,4,HIGH_VISIBILITY_ZERO_CLICKS,keyword article,informational,"Review search intent, title/snippet, and infor..."
4,5,HIGH_VISIBILITY_ZERO_CLICKS,keyword article,informational,"Review search intent, title/snippet, and infor..."
5,6,HIGH_VISIBILITY_ZERO_CLICKS,keyword article,commercial,"Review search intent, title/snippet, and infor..."
6,7,HIGH_VISIBILITY_ZERO_CLICKS,keyword article,commercial,"Review search intent, title/snippet, and infor..."
7,8,HIGH_VISIBILITY_ZERO_CLICKS,keyword article,informational,"Review search intent, title/snippet, and infor..."
8,9,HIGH_VISIBILITY_ZERO_CLICKS,keyword article,commercial,"Review search intent, title/snippet, and infor..."
9,10,HIGH_VISIBILITY_ZERO_CLICKS,keyword article,transactional,"Review search intent, title/snippet, and infor..."


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

This playbook is intended to help SEO and content teams prioritize pages for human review.

The ranked queue highlights pages that match observed opportunity patterns in the March 2026 modeling window. Reason codes, content type, and search intent provide context for deciding what should be reviewed first.

The queue is a decision-support tool rather than an automated content optimizer. A high-ranked page should be treated as a review candidate, not as proof that changing the page will improve clicks or traffic.

### Limits

The opportunity label is a proxy based on high impressions and zero observed clicks in the modeling window. It is not a direct measure of future traffic gain or business value.

The Week-6 validation audit also identified leakage risks in some Week-5 model features. Therefore, the Week-7 playbook does not treat the Week-5 model score as a causal or guaranteed performance estimate.

The data represents an observed historical window, so the recommendations may not generalize to future periods, new clients, or substantially different content populations.

Content type and search intent provide useful review context, but they do not determine the correct editorial action automatically.

Human review remains necessary before any content change is made.

In [ ]:
# ============================================================
# SECTION 2 — Intended use and limits
# ============================================================

# Summarize the population covered by the playbook.

intended_use_summary = pd.DataFrame({
    "Measure": [
        "Queue rows",
        "Unique clients",
        "Opportunity candidates",
        "Opportunity rate",
        "Modeling window",
        "Primary use"
    ],
    "Observed_value": [
        len(queue),
        queue["client_hash_id"].nunique(),
        int(queue["opportunity_label"].sum()),
        round(queue["opportunity_label"].mean(), 6),
        "March 2026",
        "Human-reviewed content prioritization"
    ]
})

display(intended_use_summary)

print(
    "\nInterpretation: the queue is a decision-support shortlist "
    "for human review, not an automated content-action system."
)



,Measure,Observed_value
0,Queue rows,175304
1,Unique clients,47
2,Opportunity candidates,4057
3,Opportunity rate,0.023143
4,Modeling window,March 2026
5,Primary use,Human-reviewed content prioritization



Interpretation: the queue is a decision-support shortlist for human review, not an automated content-action system.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review rules

Every high-priority page should be reviewed by a person before any content change is made.

The reviewer should check:

1. Whether the page matches the intended search intent.
2. Whether the title and snippet accurately represent the page.
3. Whether the content is relevant and useful for the query.
4. Whether the page is technically accessible and indexable.
5. Whether the observed zero-click or low-CTR pattern is persistent rather than a temporary fluctuation.
6. Whether the page has business or editorial constraints that are not represented in the dataset.
7. Whether the proposed change is appropriate for the content type.

The queue should help a reviewer decide where to look first. It should not decide what content must be changed.

### No-go list

The following actions should not be automated from this analysis:

- Automatically rewriting titles, snippets, or page content.
- Automatically deleting or redirecting pages.
- Automatically changing search intent or content type.
- Automatically publishing content changes.
- Automatically declaring that a page will gain clicks or traffic after a change.
- Automatically making decisions based only on the opportunity score.
- Automatically acting on pages when important metadata or search-intent information is missing.

These actions require human judgment and, where appropriate, additional evidence or testing.

In [ ]:
# ============================================================
# SECTION 3 — Human review + no-go checks
# ============================================================

# Identify queue items that require human review.
queue["human_review_required"] = True

# Pages with missing intent need an additional review flag.
queue["intent_review_flag"] = queue["main_intent"].isna()

# Pages with an unknown content type also require additional review.
queue["archetype_review_flag"] = queue["content_type"].isna()

# Summarize review requirements.
review_summary = pd.DataFrame({
    "Measure": [
        "Total queue rows",
        "Rows requiring human review",
        "Rows with missing search intent",
        "Rows with missing content type",
        "High-priority zero-click opportunities"
    ],
    "Observed_value": [
        len(queue),
        int(queue["human_review_required"].sum()),
        int(queue["intent_review_flag"].sum()),
        int(queue["archetype_review_flag"].sum()),
        int(
            (queue["reason_code"] == "HIGH_VISIBILITY_ZERO_CLICKS")
            .sum()
        )
    ]
})

display(review_summary)

# Safety check:
# no row should be treated as automatically approved for action.
assert queue["human_review_required"].all()

print(
    "\nSafety check passed: every queue item remains subject "
    "to human review before action."
)

,Measure,Observed_value
0,Total queue rows,175304
1,Rows requiring human review,175304
2,Rows with missing search intent,15296
3,Rows with missing content type,0
4,High-priority zero-click opportunities,4057



Safety check passed: every queue item remains subject to human review before action.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring and retrain triggers

The playbook should be monitored because the observed opportunity pattern may change over time.

The following signals should be checked regularly:

- The opportunity rate changes substantially from the current observed rate of 2.31%.
- The distribution of impressions or average position changes materially.
- The share of high-visibility zero-click pages changes substantially.
- Content-type or search-intent distributions change.
- The relationship between visibility and clicks changes over time.
- Reviewers repeatedly reject the recommended actions, indicating that the reason codes are no longer useful.

A model or ranking system should be reconsidered or retrained when these patterns persist across monitoring periods rather than because of a single unusual observation.

For this analysis, monitoring thresholds are decision triggers rather than claims about production performance. Future retraining should use a fresh, time-separated evaluation period and repeat the leakage and grouped-validation checks from Week 6.

In [ ]:
# ============================================================
# SECTION 4 — Monitoring / retrain triggers
# ============================================================

current_opportunity_rate = queue["opportunity_label"].mean()

reason_distribution = (
    queue["reason_code"]
    .value_counts(normalize=True)
    .rename("share")
    .reset_index()
    .rename(columns={"index": "reason_code"})
)

content_distribution = (
    queue["content_type"]
    .value_counts(normalize=True, dropna=False)
    .rename("share")
    .reset_index()
    .rename(columns={"index": "content_type"})
)

monitoring_summary = pd.DataFrame({
    "Metric": [
        "Current opportunity rate",
        "High-visibility zero-click share",
        "High-visibility review share",
        "Monitor share",
        "Rows with missing search intent"
    ],
    "Observed_value": [
        current_opportunity_rate,
        (
            queue["reason_code"]
            .eq("HIGH_VISIBILITY_ZERO_CLICKS")
            .mean()
        ),
        (
            queue["reason_code"]
            .eq("HIGH_VISIBILITY_REVIEW")
            .mean()
        ),
        (
            queue["reason_code"]
            .eq("MONITOR")
            .mean()
        ),
        queue["main_intent"].isna().mean()
    ]
})

display(monitoring_summary)

print("\nMonitoring rule:")
print(
    "Reassess the playbook if these distributions change materially "
    "and persist across future monitoring periods."
)

print(
    "\nRetrain rule:"
)
print(
    "Retrain/revalidate only after collecting a fresh time-separated "
    "evaluation period and repeating leakage and grouped-validation audits."
)

,Metric,Observed_value
0,Current opportunity rate,0.023143
1,High-visibility zero-click share,0.023143
2,High-visibility review share,0.226932
3,Monitor share,0.749926
4,Rows with missing search intent,0.087254



Monitoring rule:
Reassess the playbook if these distributions change materially and persist across future monitoring periods.

Retrain rule:
Retrain/revalidate only after collecting a fresh time-separated evaluation period and repeating leakage and grouped-validation audits.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exports for the paper

The ranked queue and supporting summary metrics are exported so that the next stage of the research paper can reuse the same evidence.

The queue is regenerated by the notebook rather than treated as a manually maintained production dataset. The exported results are intended for analysis and paper development, not production deployment.

In [ ]:
# ============================================================
# SECTION 5 — Exports for the paper
# ============================================================

from pathlib import Path
import json

# Create required output directories.
output_dir = Path("work/outputs")
figure_dir = Path("work/figures")

output_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Export ranked action queue
# ------------------------------------------------------------

queue_path = output_dir / "w07_ranked_action_queue.csv"

# Keep the export useful for analysis while avoiding private
# client/content identifiers.
export_columns = [
    "rank",
    "priority_level",
    "reason_code",
    "suggested_action",
    "archetype_action",
    "content_type",
    "main_intent",
    "total_impressions",
    "total_clicks",
    "avg_position",
    "ctr",
    "opportunity_label",
    "human_review_required",
    "intent_review_flag",
    "archetype_review_flag"
]

queue_export = queue[export_columns].copy()

queue_export.to_csv(queue_path, index=False)

# ------------------------------------------------------------
# 2. Export paper metrics
# ------------------------------------------------------------

metrics = {
    "dataset_rows": int(len(queue)),
    "unique_clients": int(queue["client_hash_id"].nunique()),
    "opportunity_candidates": int(queue["opportunity_label"].sum()),
    "opportunity_rate": float(queue["opportunity_label"].mean()),
    "impression_threshold": float(impression_threshold),
    "high_visibility_zero_click_share": float(
        queue["reason_code"]
        .eq("HIGH_VISIBILITY_ZERO_CLICKS")
        .mean()
    ),
    "high_visibility_review_share": float(
        queue["reason_code"]
        .eq("HIGH_VISIBILITY_REVIEW")
        .mean()
    ),
    "monitor_share": float(
        queue["reason_code"]
        .eq("MONITOR")
        .mean()
    ),
    "missing_intent_share": float(
        queue["main_intent"].isna().mean()
    ),
    "modeling_window": "March 2026",
    "intended_use": "Human-reviewed content prioritization"
}

metrics_path = output_dir / "w07_playbook_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

# ------------------------------------------------------------
# 3. Verify exports
# ------------------------------------------------------------

assert queue_path.exists()
assert metrics_path.exists()

print("Exports created successfully:")
print("-", queue_path)
print("-", metrics_path)

print("\nQueue export rows:", len(queue_export))
print("Queue export columns:", len(queue_export.columns))

Exports created successfully:
- work/outputs/w07_ranked_action_queue.csv
- work/outputs/w07_playbook_metrics.json

Queue export rows: 175304
Queue export columns: 15


## Self-check

Before you submit, confirm each line honestly:

- [✔️] Every section above is filled — markdown thinking AND the code that backs it
- [✔️] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔️] No client names, URLs, or private queries anywhere
- [✔️] My claims use careful words: observed, measured, directional, decision-support
- [✔️] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.